Battery Voltage Prediction using Machine Learning (Linear Regression Baseline)

## 1. Problem Definition

This project aims to develop a machine learning model to predict battery voltage using environmental parameters, including:
- Temperature
- Humidity
- Solar radiation

The objective is to support predictive maintenance of off-grid power supply systems in Automatic Weather Stations (AWS).

In [ ]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv('data/Batt_Sample.csv')

# Convert datetime
df['date_time'] = pd.to_datetime(df['date_time'], errors='coerce')

print("Data Types:\n", df.dtypes)
print("Shape:", df.shape)

## 2. Data Preprocessing
Handling missing values using interpolation and forward/backward filling.

In [ ]:
# Define features and target
cols_X = ['temp_avg', 'rh_avg', 'sr_avg']
col_y = ['batt_volt']

# Check missing
print("Missing values:\n", df.isna().sum())

# Interpolate input features
df[cols_X] = df[cols_X].interpolate(method='linear', limit=3)
df[cols_X] = df[cols_X].bfill().ffill()

# Drop missing target
df = df.dropna(subset=col_y)

# Drop any remaining NaN
df = df.dropna()

print("Remaining missing:\n", df.isna().sum())
print("Clean shape:", df.shape)

## 3. Feature Engineering (Sliding Window)
Using past 4 time steps to predict next voltage value.

In [ ]:
data = df[cols_X].values
output = df[col_y].values

window_size = 4

X, y = [], []

for i in range(len(data) - window_size):
    X.append(data[i:i+window_size])
    y.append(output[i+window_size])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

## 4. Train-Test Split

In [ ]:
split = int(0.8 * len(X))

X_train = X[:split]
X_test  = X[split:]

y_train = y[:split]
y_test  = y[split:]

## 5. Normalization
Standardizing features to improve model performance.

In [ ]:
# Normalize X
mean_X = np.mean(X_train, axis=(0,1))
std_X = np.std(X_train, axis=(0,1))

X_train_norm = (X_train - mean_X) / std_X
X_test_norm  = (X_test  - mean_X) / std_X

# Normalize y
mean_y = np.mean(y_train)
std_y = np.std(y_train)

y_train_norm = (y_train - mean_y) / std_y
y_test_norm  = (y_test  - mean_y) / std_y

print("Normalized shapes:")
print(X_train_norm.shape, y_train_norm.shape)

#Flatten for Linear Regression
X_train_flat = X_train_norm.reshape(X_train_norm.shape[0], -1)
X_test_flat  = X_test_norm.reshape(X_test_norm.shape[0], -1)

## 6. Model Training
Using Linear Regression as baseline model.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_flat, y_train_norm)

## 7. **Prediction and Evaluation**

In [ ]:
from sklearn.metrics import mean_squared_error

# Prediction (normalized)
y_pred_norm = model.predict(X_test_flat)

# MSE (normalized)
mse = mean_squared_error(y_test_norm, y_pred_norm)
print("MSE (normalized):", mse)

# Convert back to real scale
y_pred = y_pred_norm * std_y + mean_y
y_test_real = y_test_norm * std_y + mean_y

rmse = np.sqrt(mean_squared_error(y_test_real, y_pred))
print("RMSE (real):", rmse)

# Stats
print("Target min:", y.min())
print("Target max:", y.max())
print("Target std:", std_y)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(y_test_real, label='Actual')
plt.plot(y_pred, label='Prediction')
plt.legend()
plt.title("Battery Voltage Prediction")
plt.show()

## 7. Conclusion
The model achieves moderate performance and serves as a baseline for future work includes LSTM/TCN and physics-informed models.